In [ ]:
import pandas as pd

# Path to data frame with WSIs
abmil_inference = r"D:\DATA\EXP3_abmil\abmil_inference.csv"
abmil_training = r"D:\DATA\EXP3_abmil\abmil_training.csv"
df_inference = pd.read_csv(abmil_inference)
df_training = pd.read_csv(abmil_training)

overlap = set(df_inference['rekvnr']) & set(df_training['rekvnr'])
if overlap: 
    print(f'There are {len(overlap)} overlapping values.')
else: 
    print('No overlap.')

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_training[col] = df_training[col].apply(strings2lists)
    df_inference[col] = df_inference[col].apply(strings2lists)

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 1, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 1,
    'Resection Margin Not Free': 1, 
    'Proliferative/Pre-neoplastic Changes': 1, 
    'Benign Neoplasm': 1, 
    'Uncertain / Borderline Neoplasm': 1, 
    'In Situ Neoplasm': 1, 
    'Malignant Neoplasm': 1,
}

df_training["M_idx"] = df_training["M_category"].apply(lambda lst: [class_dict[x] for x in lst])
df_inference["M_idx"] = df_inference["M_category"].apply(lambda lst: [class_dict[x] for x in lst])

# 0: Healthy
# 1: Non-healthy

In [ ]:
df_training['M_idx'] = df_training['M_idx'].apply(lambda x: max(x) if isinstance(x, list) else x)
df_inference['M_idx'] = df_inference['M_idx'].apply(lambda x: max(x) if isinstance(x, list) else x)

In [ ]:
print("Training data:")
df_training['M_idx'].value_counts()

In [ ]:
print("Inference data:")
df_inference['M_idx'].value_counts()

In [ ]:
from abmil import TrainABMILPipeline

abmil_path = r"D:\DATA\abmil_checkpoints\abmil_hopt_binary.pt"

pipeline = TrainABMILPipeline(df_training, 'filename', 'M_idx', 'features_h-optimus-0', 'tiles_224', zarr_dir, abmil_path)
# pipeline.run_pipeline(max_tiles=50000, n_epochs=100, seed=42, validation_fraction=0.10, early_stopping_patience=5)
pipeline.validate_slides()

In [ ]:
df_training_clean = pipeline.df

output_file = r"D:\DATA\EXP3_abmil\abmil_training_clean.csv"
df_training_clean.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")

In [ ]:
abmil_training_clean = r"D:\DATA\EXP3_abmil\abmil_training_clean.csv"
df_training_clean = pd.read_csv(abmil_training_clean)

from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_training_clean[col] = df_training_clean[col].apply(strings2lists)

In [ ]:
print("Training data:")
df_training_clean['M_idx'].value_counts()

In [ ]:
pipeline = TrainABMILPipeline(df_training_clean, 'filename', 'M_idx', 'features_h-optimus-0', 'tiles_224', zarr_dir, abmil_path)
pipeline.save_abmil(max_tiles=50000, seed=42, n_epochs=17)

In [ ]:
from abmil import ABMILInference

slides = df_inference["filename"].tolist()
abmil_path = r"D:\DATA\abmil_checkpoints\abmil_hopt_binary.pt"
inference_path = r"D:\DATA\abmil_checkpoints\inference_hopt_binary.pkl"

inference = ABMILInference(checkpoint_path=abmil_path, zarr_dir=zarr_dir, slides=slides, cache_path=inference_path)
inference.process_slides()

In [ ]:
results_df = inference.results_dataframe()
print(f"Results for {len(results_df)} slides:\n")
print(results_df.head())

In [ ]:
skipped_slides = inference.get_skipped_slides()
print(f"Skipped slides: {len(skipped_slides)}\n")
print(skipped_slides.head())

In [ ]:
# Save to csv
output_file = r"D:\DATA\abmil_checkpoints\inference_hopt_binary.csv"
results_df.to_csv(output_file, index=False)

output_file = r"D:\DATA\abmil_checkpoints\inference_hopt_binary_skipped.csv"
skipped_slides.to_csv(output_file, index=False)

In [ ]:
from helper_functions import lists2tuples
    
df_inference = lists2tuples(df_inference)

result = r"D:\DATA\abmil_inference\inference_hopt_binary.csv"
results_df= pd.read_csv(result)

In [ ]:
from abmil import ABMILEvaluation

# Initialize evaluator if you have ground truth labels
evaluator = ABMILEvaluation(results_df, metadata_df = df_inference, true_label_col="M_idx")

# Match predictions with ground truth
matched_df = evaluator.match_true_labels(slide_id_col="filename", results_path_col="slide_path")
print(f"Matched {len(matched_df)} slides with ground truth")
print(matched_df.head())

In [ ]:
matched_df['M_idx'].value_counts()

In [ ]:
# Compute metrics
metrics = evaluator.compute_metrics()
print(f"\nMacro AUC: {metrics['auc']:.4f}")
print(f"Per-class AUC: {metrics['per_class_aucs']}")

# Confusion matrix and classification report
evaluator.assessment_report()

In [ ]:
# Plot precision-recall curves
evaluator.pr_curves()

In [ ]:
# Plot ROC curves for all classes
evaluator.ovr_roc_curves()

In [ ]:
# Group by category
group_metrics = evaluator.group_by_metrics("T_category")

In [ ]:
slide = slides[0]  # Change index to select a different slide
inference.attention_heatmap(slide)

In [ ]:
# Training QC
# dataframe with feature norms and bagsize

import os
import numpy as np
import pandas as pd
from wsidata import open_wsi

key = "features_h-optimus-0"

slides = df_training_clean["filename"].tolist()
label_map = df_training_clean.groupby("filename")["M_idx"].max().to_dict()

rows = []

for slide_path in paths:
    print(slide_path)
    zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))

    try:
        wsi = open_wsi(slide_path, zarr_path)
    except Exception as e:
        print(e)
        continue

    label = label_map.get(slide_path)

    X = wsi.tables.get(key, {}).X if key in wsi.tables else None
    if X is None or X.size == 0:
        continue

    norms = np.linalg.norm(X, axis=1)
    bag_size = X.shape[0]

    rows.extend([{
        "filename": slide_path,
        "M_idx": label,
        "bag_size": bag_size,
        "feature_norm": float(n),
    } for n in norms
    ])

df_norm = pd.DataFrame(rows)

In [ ]:
norm_summary = (
    df_norm
    .groupby("feature_norm")
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

print("Norm Summary: \n", norm_summary)

bag_summary = (
    df_norm
    .groupby("M_idx")["bag_size"]
    .agg(["count", "mean", "std", "median"])
    .round(2)
)

print("Bag Summary: \n", bag_summary)

In [ ]:
m_idx_map = {
    0: "0 - Normal (n=)",
    1: "1 - Non-healthy (n=)"
}

df_norm["M_label"] = df_norm["M_idx"].map(m_idx_map)

In [ ]:
# Plot, bagsize by M idx

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

order = ["0 - Normal (n=)", "1 - Non-healthy (n=)"]

palette = sns.color_palette(n_colors=len(order))
color_map = dict(zip(order, palette))

sns.set_style("white")

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=df_norm,
    x="bag_size",
    hue="M_label",
    hue_order=order,
    bins=50,
    multiple="stack",
    alpha=0.4,
    palette=color_map,
    ax=ax
)

legend = ax.get_legend()
legend.set_title("M idx")

handles = legend.legend_handles
ymax = ax.get_ylim()[1]

for i, label in enumerate(order):
    values = bag_df.loc[bag_df["M_label"] == label, "bag_size"]

    mean_val = values.mean()
    median_val = values.median()
    color = color_map[label]

    ax.axvline(mean_val, color=color, linestyle="-", linewidth=2)
    ax.axvline(median_val, color=color, linestyle="--", linewidth=2)

    # Mean
    ax.text(
        mean_val,
        ymax * 0.9,
        f"Mean: {mean_val:.1f}",
        color=color,
        ha='left',
        fontsize=8, 
        backgroundcolor='white'
    )
    
    # Median
    ax.text(
        median_val,
        ymax * 0.75,
        f"Median: {median_val:.1f}",
        color=color,
        ha='left',
        fontsize=8,
        fontweight='bold',
        backgroundcolor='white'
    )

ax.set_title("Bag Size Distribution")
ax.set_xlabel("Instances per Slide")
ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Plot, raw and normalized feature norms

df_norm["norm_zscore"] = df_plot.groupby(["filename"])["feature_norm"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw norms
sns.histplot(
    data=df_plot,
    x="feature_norm",
    bins=100,
    alpha=0.4,
    ax=axes[0],
    legend= True
)
legend = ax.get_legend()
legend.set_title("M idx")
axes[0].set_title("Feature Norm Distribution")
axes[0].set_xlabel("L2 Norm")

# Normalized norms
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    bins=100,
    alpha=0.4,
    ax=axes[1],
)
axes[1].set_title("Normalized Norms (Per-Slide)")
axes[1].set_xlabel("Z-scored Norm")

plt.tight_layout()
plt.show()

In [ ]:
# Plot, mean feature norm by bag size
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate mean feature norm per slide
df_agg = df_norm.groupby("filename").agg({
    "feature_norm": "mean",
    "bag_size": "first",
    "M_idx": "first"
}).reset_index()

# Map M_idx to labels
m_idx_map = {
    0: "0 - Normal",
    1: "1 - Non-healthy",
}
df_agg["M_label"] = df_agg["M_idx"].map(m_idx_map)

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_agg,
    x="bag_size",
    y="feature_norm",
    hue="M_label",
    s=100,
    alpha=0.6,
    ax=ax
)

ax.set_xlabel("Bag Size")
ax.set_ylabel("Mean Feature Norm (L2)")
ax.set_title("Mean Feature Norm vs. Bag Size by M idx")
ax.legend(title="Morphology Category", bbox_to_anchor=(1.05, 1), loc='upper left')

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
from visualize_features import FeatureVisualizer

categories = ["T_category", "T_text", "M_category", "M_text", "team", "sex", "alder", "alder gruppe", 'mattype tekst', 'stain', 
              'snomed_code', 'snomed_text', 'undersoeger_anonymous', 'M_idx'] 

model = 'h-optimus-0'
feat_col = f"features_{model}"
for cat in categories:
    try:
        print(f"\n Visualizing model: {model}, feature: {cat}")
        viz = FeatureVisualizer(df_training_clean, zarr_dir = zarr_dir, cache_path = cache_features, model = model, label_col = cat, aggregated = True)
        viz.plot_embeddings(method =  "tsne", mode = "aggregated", every_nth = 1)
           
    except Exception as e:
        print(f"Error visualizing projection: {e}")
        continue